<div align="center">

# Universidad de Sevilla  
## Grado en Ingeniería del Software  
### Escuela Técnica Superior de Ingeniería Informática  
### Inteligencia Artificial (IA) – Curso 2025/2026  

<img src="../../img/portada.png" width="300"/>

---

#  Aprendizaje automático relacional  
### Primera Convocatoria – Junio 2026  

**Profesor:** Pedro Almagro Blanco

**Autores:**  
David Espina Apellaniz  
Marco Padilla Gómez  

**Fecha de entrega:** 8 de Junio de 2025  

</div>

# 05 - kNN: experimentos de hiperparámetros

En este notebook se prueban distintas configuraciones del modelo **kNN**.

Se comparan distintos valores de número de vecinos, pesos y métricas de distancia para seleccionar la mejor combinación según `f1`.

## Objetivo metodologico

Este cuaderno analiza la sensibilidad de kNN a sus hiperparametros principales: numero de vecinos, ponderacion de vecinos y metrica de distancia.


## 1. Importación de librerías y configuración inicial

Se importan las funciones de carga, evaluación, guardado y los componentes necesarios para construir el `Pipeline` de kNN.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sys.path.append("../../")

from src.data_loader import load_processed_dataset
from src.evaluation import evaluate_model, save_metrics, save_model
from src.model_training import split_data

ROOT = Path("../../").resolve()
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Entrenamiento con distintas configuraciones

Se prueban combinaciones de `n_neighbors`, `weights` y `metric` para evaluar cuál funciona mejor sobre el conjunto de test.

### Hiperparametros evaluados

`n_neighbors` controla cuantos ejemplos influyen en la prediccion. `weights='distance'` da mas peso a vecinos cercanos. La comparacion entre distancia euclidea y Manhattan permite comprobar que nocion de cercania funciona mejor con estas variables.


In [2]:
df = load_processed_dataset(DATA_PROCESSED / "twitch_mature_features.csv")
X = df.drop(columns=["new_id", "mature"])
y = df["mature"]
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)
experiments = []
for n_neighbors in [3, 5, 7, 9]:
    for weights in ["uniform", "distance"]:
        for metric in ["euclidean", "manhattan"]:
            candidate = Pipeline([
                ("scaler", StandardScaler()),
                ("knn", KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, metric=metric)),
            ])
            candidate.fit(X_train, y_train)
            experiments.append({
                "model": "knn",
                "n_neighbors": n_neighbors,
                "weights": weights,
                "metric": metric,
                **evaluate_model(y_test, candidate.predict(X_test))
            })
results_df = pd.DataFrame(experiments)
best_row = results_df.sort_values("f1", ascending=False).iloc[0]
best_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=int(best_row["n_neighbors"]),
        weights=best_row["weights"],
        metric=best_row["metric"],
    )),
])
best_model.fit(X_train, y_train)
results_df.sort_values("f1", ascending=False)

,model,n_neighbors,weights,metric,accuracy,precision,recall,f1
0,knn,3,uniform,euclidean,0.669892,0.413793,0.308824,0.353684
2,knn,3,distance,euclidean,0.667742,0.408867,0.305147,0.349474
6,knn,5,distance,euclidean,0.684946,0.437126,0.268382,0.332574
4,knn,5,uniform,euclidean,0.688172,0.444444,0.264706,0.331797
10,knn,7,distance,euclidean,0.696774,0.466216,0.253676,0.328571
3,knn,3,distance,manhattan,0.658065,0.385000,0.283088,0.326271
7,knn,5,distance,manhattan,0.682796,0.430303,0.261029,0.324943
5,knn,5,uniform,manhattan,0.684946,0.434783,0.257353,0.323326
8,knn,7,uniform,euclidean,0.695699,0.462069,0.246324,0.321343
1,knn,3,uniform,manhattan,0.659140,0.383420,0.272059,0.318280


## 3. Guardado de resultados y mejor modelo

Se guardan las métricas de los experimentos y el mejor modelo kNN.

In [3]:
save_metrics(results_df.to_dict(orient="records"), RESULTS_DIR / "knn_grid_metrics.csv")
save_model(best_model, ROOT / "models" / "knn_grid.joblib")

## 4. Conclusiones

El rendimiento de kNN depende especialmente del número de vecinos y de la métrica de distancia. Estos experimentos permiten seleccionar una configuración más adecuada que la versión base.